# 04 — Chunking

**Scenario.** You cannot embed a whole policy vault as one vector. You split files into **chunks** (passages) and keep **metadata** (classification, status, file type) on each chunk.

If you split in the middle of a table, the model loses the column names. This demo splits markdown/HTML on headings, CSV as one chunk per row, and keeps a header on every embedding.

1. Load the mixed corpus.
2. Chunk it.
3. Print one markdown chunk, one CSV row, one TSX chunk.

Context:

- A chunk is the unit retrieved and cited.
- Metadata travels with every chunk: classification, status, source file, type.
- The embedded text includes a small header so table rows and code snippets keep context.

Edge cases:

- Splitting tables without headers creates orphan values.
- Splitting code without comments can hide security intent.
- A chunk can be relevant but still forbidden for a role.


Definitions and context:

| Concept | Meaning |
| --- | --- |
| Chunk | Small passage indexed, retrieved, and cited |
| Chunk boundary | Where one passage ends and the next begins |
| Metadata | Structured fields stored with a chunk |
| Contextual header | Extra text prepended before embedding |
| Source type | Original file format such as md, pdf, csv, tsx |

Chunking is one of the most important RAG design choices. A chunk should be
small enough to retrieve precisely, but large enough to preserve meaning.
Metadata is not optional in high-security settings: classification and status
decide whether a chunk can be shown, even if it is relevant.


In [1]:
import _course
from collections import Counter
from lib.ingest import load_documents
from lib.pipeline import chunk_documents

docs = load_documents()
chunks = chunk_documents(docs)
print("documents:", len(docs), "chunks:", len(chunks))
print("by type:", dict(Counter(d.source_type for d in docs)))
print()
for wanted in ("md", "csv", "tsx"):
    c = next(ch for ch in chunks if ch.source_type == wanted)
    print("---", c.source_type, c.doc_id, "/", c.section, "---")
    print(c.embedded_text[:500])
    print()


course: Northline secure RAG tutorial
llm   : False | model: grok-4.6
documents: 16 chunks: 51
by type: {'tsx': 2, 'md': 5, 'pdf': 2, 'yaml': 1, 'json': 1, 'csv': 3, 'html': 2}

--- md POL-AI-USE-2026 / Acceptable Use of Internal AI Assistants ---
Document: Acceptable Use of Internal AI Assistants (POL-AI-USE-2026)
Source: policies/acceptable-use-ai.md (md)
Section: Acceptable Use of Internal AI Assistants
Classification: internal
Effective: 2026-02-01 Status: current

Staff may use the Northline knowledge assistant for policy lookup, control
interpretation, and runbook navigation. It is not a client channel and not a
system of record.

--- csv TBL-RETENTION-2026 / Table overview ---
Document: Records retention schedule (TBL-RETENTION-2026)
Source: tables/retention-schedule.csv (csv)
Section: Table overview
Classification: internal
Effective: 2026-03-10 Status: current

Columns: record, retention, system_of_record
record=Policy PDFs (current and superseded) | retention=7 years after wi

Next: `05-mixed-sources.ipynb` — why a bank corpus is never only markdown.
